# Explore here

Step 1: Loading the dataset

In [22]:
import pandas as pd

total_data = pd.read_csv("https://raw.githubusercontent.com/4GeeksAcademy/naive-bayes-project-tutorial/main/playstore_reviews.csv")

total_data.head()

,package_name,review,polarity
0,com.facebook.katana,privacy at least put some option appear offli...,0
1,com.facebook.katana,"messenger issues ever since the last update, ...",0
2,com.facebook.katana,profile any time my wife or anybody has more ...,0
3,com.facebook.katana,the new features suck for those of us who don...,0
4,com.facebook.katana,forced reload on uploading pic on replying co...,0


In [23]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
import json
import os

# 1. Cargar el dataset
df = pd.read_csv("https://raw.githubusercontent.com/4GeeksAcademy/naive-bayes-project-tutorial/main/playstore_reviews.csv")

# 2. Preprocesar (borrar columna que no necesitamos, limpiar texto)
df = df.drop("package_name", axis=1)
df["review"] = df["review"].str.strip().str.lower()

# 3. Vectorizar
vectorizer = CountVectorizer(stop_words="english")
X = vectorizer.fit_transform(df["review"])

# 4. Obtener vocabulario y convertir a lista
vocab = vectorizer.get_feature_names_out()

# 5. Convertir la matriz en una lista de diccionarios por fila (documento)
factorized_reviews = []
for i in range(X.shape[0]):
    row_dict = {}
    for idx in X[i].nonzero()[1]:  # solo columnas donde el valor no es 0
        word = vocab[idx]
        count = X[i, idx]
        row_dict[word] = int(count)
    factorized_reviews.append(row_dict)

# 6. Guardar como JSON
os.makedirs("data", exist_ok=True)

with open("data/reviews_factorized.json", "w", encoding="utf-8") as f:
    json.dump(factorized_reviews, f, indent=2, ensure_ascii=False)

print("✔️ Factorización guardada en 'data/reviews_factorized.json'")


✔️ Factorización guardada en 'data/reviews_factorized.json'


In [24]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
import json
import os

# 1. Cargar y preparar el dataset
df = pd.read_csv("https://raw.githubusercontent.com/4GeeksAcademy/naive-bayes-project-tutorial/main/playstore_reviews.csv")
df = df.drop("package_name", axis=1)
df["review"] = df["review"].str.strip().str.lower()

# 2. Vectorizar los textos
vectorizer = CountVectorizer(stop_words="english", max_features=1000)  # límite para simplificar
X = vectorizer.fit_transform(df["review"]).toarray()
df_vectorized = pd.DataFrame(X, columns=vectorizer.get_feature_names_out())

# 3. Duplicar dataframes
df_con_outliers = df_vectorized.copy()
df_sin_outliers = df_vectorized.copy()

# 4. Función para reemplazar outliers
def replace_outliers_from_column(column, df):
    stats = df[column].describe()
    iqr = stats["75%"] - stats["25%"]
    upper_limit = stats["75%"] + 1.5 * iqr
    lower_limit = stats["25%"] - 1.5 * iqr
    if lower_limit < 0: lower_limit = 0
    df[column] = df[column].apply(lambda x: x if (x <= upper_limit) else upper_limit)
    df[column] = df[column].apply(lambda x: x if (x >= lower_limit) else lower_limit)
    return df.copy(), [lower_limit, upper_limit]

# 5. Reemplazar y guardar límites
outliers_dict = {}

for column in df_vectorized.columns:
    df_sin_outliers, limits = replace_outliers_from_column(column, df_sin_outliers)
    outliers_dict[column] = limits

# 6. Guardar el diccionario de outliers como JSON
os.makedirs("data", exist_ok=True)

with open("data/outliers_dict.json", "w", encoding="utf-8") as f:
    json.dump(outliers_dict, f, indent=2)

print("✔️ Diccionario de outliers guardado en 'data/outliers_dict.json'")


✔️ Diccionario de outliers guardado en 'data/outliers_dict.json'


In [25]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import os

# Crear carpeta de salida
os.makedirs("data/processed", exist_ok=True)

# Variable objetivo
y = df["polarity"]

# 1. Separar variables predictoras (X) de la target (y)
X_con_outliers = df_con_outliers
X_sin_outliers = df_sin_outliers

# 2. Dividir train/test
X_train_con_outliers, X_test_con_outliers, y_train, y_test = train_test_split(X_con_outliers, y, test_size=0.2, random_state=42)
X_train_sin_outliers, X_test_sin_outliers = train_test_split(X_sin_outliers, test_size=0.2, random_state=42)

# 3. Escalar (usamos StandardScaler como ejemplo)
scaler = StandardScaler()

X_train_con_outliers_scaled = scaler.fit_transform(X_train_con_outliers)
X_test_con_outliers_scaled = scaler.transform(X_test_con_outliers)

X_train_sin_outliers_scaled = scaler.fit_transform(X_train_sin_outliers)
X_test_sin_outliers_scaled = scaler.transform(X_test_sin_outliers)

# 4. Convertir de nuevo a DataFrame
X_train_con_outliers_scaled = pd.DataFrame(X_train_con_outliers_scaled, columns=X_con_outliers.columns)
X_test_con_outliers_scaled = pd.DataFrame(X_test_con_outliers_scaled, columns=X_con_outliers.columns)

X_train_sin_outliers_scaled = pd.DataFrame(X_train_sin_outliers_scaled, columns=X_sin_outliers.columns)
X_test_sin_outliers_scaled = pd.DataFrame(X_test_sin_outliers_scaled, columns=X_sin_outliers.columns)

# 5. Guardar como archivos .xlsx
X_train_con_outliers_scaled.to_excel("data/processed/X_train_con_outliers.xlsx", index=False)
X_test_con_outliers_scaled.to_excel("data/processed/X_test_con_outliers.xlsx", index=False)

X_train_sin_outliers_scaled.to_excel("data/processed/X_train_sin_outliers.xlsx", index=False)
X_test_sin_outliers_scaled.to_excel("data/processed/X_test_sin_outliers.xlsx", index=False)

y_train.to_excel("data/processed/y_train.xlsx", index=False)
y_test.to_excel("data/processed/y_test.xlsx", index=False)

print("✔️ Datasets escalados y guardados correctamente.")


✔️ Datasets escalados y guardados correctamente.


In [29]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, MaxAbsScaler, RobustScaler
import pandas as pd
import os

# Crear carpeta de salida
os.makedirs("data/processed", exist_ok=True)

# Variable objetivo
y = df["polarity"]

# Dataset con y sin outliers
X_con_outliers = df_con_outliers
X_sin_outliers = df_sin_outliers

# Dividir en train/test
X_train_con, X_test_con, y_train, y_test = train_test_split(X_con_outliers, y, test_size=0.2, random_state=42)
X_train_sin, X_test_sin = train_test_split(X_sin_outliers, test_size=0.2, random_state=42)

# Diccionario de escaladores
scalers = {
    "minmax": MinMaxScaler(),
    "maxabs": MaxAbsScaler(),
    "robust": RobustScaler()
}

# Escalar, reconstruir DataFrames y guardar
for name, scaler in scalers.items():
    # Con outliers
    X_train_con_scaled = scaler.fit_transform(X_train_con)
    X_test_con_scaled = scaler.transform(X_test_con)
    
    df_train_con = pd.DataFrame(X_train_con_scaled, columns=X_train_con.columns)
    df_test_con = pd.DataFrame(X_test_con_scaled, columns=X_test_con.columns)
    
    df_train_con.to_excel(f"data/processed/X_train_con_outliers_{name}.xlsx", index=False)
    df_test_con.to_excel(f"data/processed/X_test_con_outliers_{name}.xlsx", index=False)
    
    # Sin outliers
    X_train_sin_scaled = scaler.fit_transform(X_train_sin)
    X_test_sin_scaled = scaler.transform(X_test_sin)

    df_train_sin = pd.DataFrame(X_train_sin_scaled, columns=X_train_sin.columns)
    df_test_sin = pd.DataFrame(X_test_sin_scaled, columns=X_train_sin.columns)
    
    df_train_sin.to_excel(f"data/processed/X_train_sin_outliers_{name}.xlsx", index=False)
    df_test_sin.to_excel(f"data/processed/X_test_sin_outliers_{name}.xlsx", index=False)

# Guardar y_train y y_test (no cambian con el escalado)
y_train.to_excel("data/processed/y_train.xlsx", index=False)
y_test.to_excel("data/processed/y_test.xlsx", index=False)

print("✔️ Datasets escalados y guardados con MinMax, MaxAbs y RobustScaler.")


✔️ Datasets escalados y guardados con MinMax, MaxAbs y RobustScaler.


In [21]:
from sklearn.preprocessing import StandardScaler
import pickle
import os

# Crear carpeta de salida para normalizadores
os.makedirs("models", exist_ok=True)

# Normalizador para datos CON outliers
normalizador_con_outliers = StandardScaler()
normalizador_con_outliers.fit(X_train_con_outliers)

with open("models/normalizador_con_outliers.pkl", "wb") as file:
    pickle.dump(normalizador_con_outliers, file)

# Normalizador para datos SIN outliers
normalizador_sin_outliers = StandardScaler()
normalizador_sin_outliers.fit(X_train_sin_outliers)

with open("models/normalizador_sin_outliers.pkl", "wb") as file:
    pickle.dump(normalizador_sin_outliers, file)

print("✔️ Normalizadores guardados en la carpeta 'models'")




✔️ Normalizadores guardados en la carpeta 'models'


In [27]:
import pickle

# Cargar el normalizador entrenado
with open("models/normalizador_con_outliers.pkl", "rb") as file:
    normalizador_cargado = pickle.load(file)

# Aplicarlo a tus datos (ejemplo con X_test_con_outliers)
X_test_con_outliers_scaled = normalizador_cargado.transform(X_test_con_outliers)


In [28]:
# Para datos sin outliers
with open("models/normalizador_sin_outliers.pkl", "rb") as file:
    normalizador_sin = pickle.load(file)

X_test_sin_outliers_scaled = normalizador_sin.transform(X_test_sin_outliers)


Step 2: Study of variables and their content

Removing spaces and converting the text to lowercase

In [2]:
def apply_preprocess(df):
    df = df.drop("package_name", axis=1)
    df["review"] = df["review"].str.strip().str.lower()

    return df

total_data = apply_preprocess(total_data)

total_data.head()

,review,polarity
0,privacy at least put some option appear offlin...,0
1,"messenger issues ever since the last update, i...",0
2,profile any time my wife or anybody has more t...,0
3,the new features suck for those of us who don'...,0
4,forced reload on uploading pic on replying com...,0


Divide the dataset into train and test

In [3]:
from sklearn.model_selection import train_test_split

X = total_data["review"]
y = total_data["polarity"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

X_train.head()

331    just did the latest update on viber and yet ag...
733    keeps crashing it only works well in extreme d...
382    the fail boat has arrived the 6.0 version is t...
704    superfast, just as i remember it ! opera mini ...
813    installed and immediately deleted this crap i ...
Name: review, dtype: object

Transform the text into a word count matrix

In [4]:
from sklearn.feature_extraction.text import CountVectorizer

vec_model = CountVectorizer(stop_words = "english")
X_train = vec_model.fit_transform(X_train).toarray()
X_test = vec_model.transform(X_test).toarray()

X_train

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(712, 3310))

Step 3: Build a naive bayes model

I select the MultinomialNB because just the target is binary while the predictors are categorical numbers.

In [5]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train, y_train)

MultinomialNB()

In [6]:
y_pred = model.predict(X_test)
y_pred

array([0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0,
       1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0,
       0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
       1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0,
       1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0,
       0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0,
       0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1,
       0, 0, 0])

In [7]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_pred)

0.8156424581005587

I will test the other sklearn Naive Bayes models:

In [8]:
from sklearn.naive_bayes import GaussianNB, BernoulliNB

for model_aux in [GaussianNB(), BernoulliNB()]:
    model_aux.fit(X_train, y_train)
    y_pred_aux = model_aux.predict(X_test)
    print(f"{model_aux} with accuracy: {accuracy_score(y_test, y_pred_aux)}")

GaussianNB() with accuracy: 0.8044692737430168
BernoulliNB() with accuracy: 0.770949720670391


I can confirm that the best model is the one I have chosen based on its theoretical foundation.



Step 4: Optimize the previous model

In [9]:
import numpy as np
from sklearn.model_selection import RandomizedSearchCV

hyperparams = {
    "alpha": np.linspace(0.01, 10.0, 200),
    "fit_prior": [True, False]
}

# We initialize the random search
random_search = RandomizedSearchCV(model, hyperparams, n_iter = 50, scoring = "accuracy", cv = 5, random_state = 42)
random_search

RandomizedSearchCV(cv=5, estimator=MultinomialNB(), n_iter=50,
                   param_distributions={'alpha': array([ 0.01      ,  0.06020101,  0.11040201,  0.16060302,  0.21080402,
        0.26100503,  0.31120603,  0.36140704,  0.41160804,  0.46180905,
        0.51201005,  0.56221106,  0.61241206,  0.66261307,  0.71281407,
        0.76301508,  0.81321608,  0.86341709,  0.91361809,  0.9638191 ,
        1.0140201 ,  1.06422111,  1.11442211,  1.1646231...
        8.54417085,  8.59437186,  8.64457286,  8.69477387,  8.74497487,
        8.79517588,  8.84537688,  8.89557789,  8.94577889,  8.9959799 ,
        9.0461809 ,  9.09638191,  9.14658291,  9.19678392,  9.24698492,
        9.29718593,  9.34738693,  9.39758794,  9.44778894,  9.49798995,
        9.54819095,  9.59839196,  9.64859296,  9.69879397,  9.74899497,
        9.79919598,  9.84939698,  9.89959799,  9.94979899, 10.        ]),
                                        'fit_prior': [True, False]},
                   random_state=42, scoring='accuracy')

In [10]:
random_search.fit(X_train, y_train)

print(f"Best hyperparameters: {random_search.best_params_}")

Best hyperparameters: {'fit_prior': False, 'alpha': np.float64(1.917638190954774)}


In [11]:
model = MultinomialNB(alpha = 1.917638190954774, fit_prior = False)
model.fit(X_train, y_train)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy_score(y_test, y_pred)

0.8212290502793296

Step 5: Save the model

In [13]:
import os
from pickle import dump

# Create the directory if it doesn't exist
os.makedirs("models", exist_ok=True)

# Save the model
dump(model, open("models/naive_bayes_alpha_1-9176382_fit_prior_False_42.sav", "wb"))
